In [31]:
%load_ext autoreload
%autoreload 2
from scripts.taxonomy_dataset import (
    taxonomy_df,
    X_train,
    X_test,
    y_train,
    y_test,
    groups_train,
    groups_test,
    transformer_train_df,
    transformer_test_df,
)
from scripts.taxonomy_dataset import (
        cross_dataset_splits,
        reason_model,
        X_train_text_embeddings,
        X_test_text_embeddings
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
taxonomy_df.keys()

Index(['dataset', 'group_id', 'trajectory_index', 'message_index',
       'current_role', 'label', 'reason', 'context_text', 'current_text',
       'is_tool_call', 'previous_messages', 'previous_tool_calls',
       'previous_tool_results', 'previous_user_messages',
       'previous_assistant_messages', 'context_char_length',
       'context_word_count', 'current_char_length', 'current_word_count',
       'reason_original', 'reason_semantic', 'taxonomy_matches',
       'failure_types', 'failure_type', 'semantic_failure_type',
       'semantic_similarity', 'final_failure_type', 'final_failure_type_v2',
       'failure_family', 'family_label'],
      dtype='object')

In [10]:
import pandas as pd
import numpy as np


print("Rows:", len(taxonomy_df))
print("Trajectories:", taxonomy_df["group_id"].nunique())

print("\nDatasets:")
print(taxonomy_df["dataset"].value_counts())

print("\nFamilies:")
print(taxonomy_df["failure_family"].value_counts())

print("\nMissing:")
print(
    taxonomy_df[
        [
            "context_text",
            "current_text",
            "failure_family",
            "family_label",
            "group_id",
        ]
    ].isna().sum()
)

print("\nFamily × dataset:")
display(
    pd.crosstab(
        taxonomy_df["dataset"],
        taxonomy_df["failure_family"],
    )
)

print("\nFamily percentages × dataset:")
display(
    (
        pd.crosstab(
            taxonomy_df["dataset"],
            taxonomy_df["failure_family"],
            normalize="index",
        ) * 100
    ).round(2)
)

Rows: 1776
Trajectories: 419

Datasets:
dataset
B    1073
C     539
A     164
Name: count, dtype: int64

Families:
failure_family
workflow_error           798
constraint_error         387
tool_use_error           275
grounding_state_error    274
reasoning_value_error     42
Name: count, dtype: int64

Missing:
context_text      0
current_text      0
failure_family    0
family_label      0
group_id          0
dtype: int64

Family × dataset:


failure_family,constraint_error,grounding_state_error,reasoning_value_error,tool_use_error,workflow_error
dataset,,,,,
A,18,45,7,24,70
B,289,93,4,148,539
C,80,136,31,103,189



Family percentages × dataset:


failure_family,constraint_error,grounding_state_error,reasoning_value_error,tool_use_error,workflow_error
dataset,,,,,
A,10.98,27.44,4.27,14.63,42.68
B,26.93,8.67,0.37,13.79,50.23
C,14.84,25.23,5.75,19.11,35.06


In [11]:
train_groups = set(groups_train)
test_groups = set(groups_test)

overlap = train_groups & test_groups

print("Train groups:", len(train_groups))
print("Test groups:", len(test_groups))
print("Overlapping groups:", len(overlap))

assert len(overlap) == 0, (
    "ERROR: trajectory leakage between train and test!"
)

Train groups: 335
Test groups: 84
Overlapping groups: 0


In [12]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\ny_train:")
print(y_train.value_counts().sort_index())

print("\ny_test:")
print(y_test.value_counts().sort_index())

print("\nTransformer train:", transformer_train_df.shape)
print("Transformer test:", transformer_test_df.shape)

print("\nTransformer columns:")
print(transformer_train_df.columns.tolist())

assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)

assert len(transformer_train_df) == len(y_train)
assert len(transformer_test_df) == len(y_test)

assert set(groups_train).isdisjoint(set(groups_test))

print("\n✓ Dataset module is internally consistent.")

X_train: (1489, 16)
X_test: (287, 16)

y_train:
family_label
0    660
1    317
2    237
3    244
4     31
Name: count, dtype: int64

y_test:
family_label
0    138
1     70
2     38
3     30
4     11
Name: count, dtype: int64

Transformer train: (1489, 15)
Transformer test: (287, 15)

Transformer columns:
['dataset', 'group_id', 'trajectory_index', 'message_index', 'current_role', 'context_text', 'current_text', 'previous_messages', 'previous_tool_calls', 'previous_tool_results', 'previous_user_messages', 'previous_assistant_messages', 'failure_family', 'family_label', 'transformer_text']

✓ Dataset module is internally consistent.


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

In [14]:
numeric_features = [
    "message_index",
    "previous_messages",
    "previous_tool_calls",
    "previous_tool_results",
    "previous_user_messages",
    "previous_assistant_messages",
]

categorical_features = [
    "current_role",
]

In [15]:
preprocessor = ColumnTransformer([
    (
        "numeric",
        StandardScaler(),
        numeric_features,
    ),

    (
        "categorical",
        OneHotEncoder(
            handle_unknown="ignore"
        ),
        categorical_features,
    ),

    (
        "current_text",
        TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True,
        ),
        "current_text",
    ),

    (
        "context_text",
        TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True,
        ),
        "context_text",
    ),
])

family_svc = Pipeline([
    (
        "preprocessor",
        preprocessor,
    ),
    (
        "classifier",
        LinearSVC(
            C=2.0,
            max_iter=20000,
            class_weight="balanced",
            random_state=42,
        ),
    ),
])

In [16]:
family_svc.fit(
    X_train,
    y_train,
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](5,)","[0,1,2,3,4]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](16,)","['message_index','previous_messages','previous_tool_calls',..., 'current_role','current_text','context_text']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,16
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (defa

In [18]:
pred = family_svc.predict(
    X_test
)

print(
    "Accuracy:",
    accuracy_score(
        y_test,
        pred,
    )
)

print(
    "Macro F1:",
    f1_score(
        y_test,
        pred,
        average="macro",
    )
)

print("\nClassification report:")

print(
    classification_report(
        y_test,
        pred,
        labels=[0, 1, 2, 3, 4],
        target_names=[
            "workflow_error",
            "constraint_error",
            "tool_use_error",
            "grounding_state_error",
            "reasoning_value_error",
        ],
        zero_division=0,
        digits=4,
    )
)

print("\nConfusion matrix:")

print(
    confusion_matrix(
        y_test,
        pred,
        labels=[0, 1, 2, 3, 4],
    )
)

Accuracy: 0.3902439024390244
Macro F1: 0.4252873498401778

Classification report:
                       precision    recall  f1-score   support

       workflow_error     0.5190    0.2971    0.3779       138
     constraint_error     0.4533    0.4857    0.4690        70
       tool_use_error     0.2273    0.3947    0.2885        38
grounding_state_error     0.2712    0.5333    0.3596        30
reasoning_value_error     0.7500    0.5455    0.6316        11

             accuracy                         0.3902       287
            macro avg     0.4442    0.4513    0.4253       287
         weighted avg     0.4473    0.3902    0.3961       287


Confusion matrix:
[[41 26 47 23  1]
 [24 34  2  9  1]
 [ 9  6 15  8  0]
 [ 5  8  1 16  0]
 [ 0  1  1  3  6]]


In [19]:
def make_taxonomy_xy(df):
    feature_cols = [
        "message_index",
        "previous_messages",
        "previous_tool_calls",
        "previous_tool_results",
        "previous_user_messages",
        "previous_assistant_messages",
        "current_role",
        "current_text",
        "context_text",
    ]

    X = df[feature_cols].copy()
    y = df["family_label"].copy()

    return X, y

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC


def make_family_svc():

    numeric_features = [
        "message_index",
        "previous_messages",
        "previous_tool_calls",
        "previous_tool_results",
        "previous_user_messages",
        "previous_assistant_messages",
    ]

    categorical_features = [
        "current_role",
    ]

    preprocessor = ColumnTransformer([
        (
            "numeric",
            StandardScaler(),
            numeric_features,
        ),

        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features,
        ),

        (
            "current_text",
            TfidfVectorizer(
                max_features=5000,
                ngram_range=(1, 2),
                min_df=2,
                sublinear_tf=True,
            ),
            "current_text",
        ),

        (
            "context_text",
            TfidfVectorizer(
                max_features=5000,
                ngram_range=(1, 2),
                min_df=2,
                sublinear_tf=True,
            ),
            "context_text",
        ),
    ])

    return Pipeline([
        (
            "preprocessor",
            preprocessor,
        ),
        (
            "classifier",
            LinearSVC(
                C=2.0,
                class_weight="balanced",
                max_iter=20000,
                random_state=42,
            ),
        ),
    ])

In [24]:
cross_results = []

for split_name, train_dfs, test_df in cross_dataset_splits:

    print("\n" + "=" * 80)
    print(split_name)
    print("=" * 80)

    train_df = pd.concat(
        train_dfs,
        ignore_index=True,
    )

    X_cross_train, y_cross_train = (
        make_taxonomy_xy(train_df)
    )

    X_cross_test, y_cross_test = (
        make_taxonomy_xy(test_df)
    )

    model = make_family_svc()

    model.fit(
        X_cross_train,
        y_cross_train,
    )

    pred = model.predict(
        X_cross_test
    )

    accuracy = accuracy_score(
        y_cross_test,
        pred,
    )

    macro_f1 = f1_score(
        y_cross_test,
        pred,
        average="macro",
        zero_division=0,
    )

    print(
        "Accuracy:",
        round(accuracy, 4),
    )

    print(
        "Macro F1:",
        round(macro_f1, 4),
    )

    print("\nClassification report:")

    print(
        classification_report(
            y_cross_test,
            pred,
            labels=[0, 1, 2, 3, 4],
            target_names=[
                "workflow_error",
                "constraint_error",
                "tool_use_error",
                "grounding_state_error",
                "reasoning_value_error",
            ],
            zero_division=0,
            digits=4,
        )
    )

    print("\nConfusion matrix:")

    print(
        confusion_matrix(
            y_cross_test,
            pred,
            labels=[0, 1, 2, 3, 4],
        )
    )

    cross_results.append({
        "experiment": split_name,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "train_rows": len(train_df),
        "test_rows": len(test_df),
    })


A+B -> C
Accuracy: 0.2208
Macro F1: 0.1766

Classification report:
                       precision    recall  f1-score   support

       workflow_error     0.3945    0.2275    0.2886       189
     constraint_error     0.1846    0.4500    0.2618        80
       tool_use_error     0.1453    0.2427    0.1818       103
grounding_state_error     0.2381    0.1103    0.1508       136
reasoning_value_error     0.0000    0.0000    0.0000        31

             accuracy                         0.2208       539
            macro avg     0.1925    0.2061    0.1766       539
         weighted avg     0.2536    0.2208    0.2128       539


Confusion matrix:
[[43 75 44 27  0]
 [13 36 19 12  0]
 [32 40 25  6  0]
 [17 31 73 15  0]
 [ 4 13 11  3  0]]

A+C -> B
Accuracy: 0.4874
Macro F1: 0.1875

Classification report:
                       precision    recall  f1-score   support

       workflow_error     0.5230    0.9276    0.6689       539
     constraint_error     0.2727    0.0104    0.0200     

In [25]:
cross_results_df = pd.DataFrame(
    cross_results
)

cross_results_df

,experiment,accuracy,macro_f1,train_rows,test_rows
0,A+B -> C,0.220779,0.176596,1237,539
1,A+C -> B,0.487418,0.187496,703,1073
2,B+C -> A,0.420732,0.239346,1612,164


In [2]:
from sklearn.linear_model import LogisticRegression

classifier = LogisticRegression(
    class_weight="balanced",
    max_iter=5000,
)

classifier.fit(
    X_train_text_embeddings,
    y_train,
)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",5000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Defaul

In [6]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
import pandas as pd


# Predict
y_pred = classifier.predict(
    X_test_text_embeddings
)


# Overall metrics
accuracy = accuracy_score(
    y_test,
    y_pred,
)

balanced_acc = balanced_accuracy_score(
    y_test,
    y_pred,
)

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
)

weighted_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted",
)


print("Accuracy:", round(accuracy, 4))
print("Balanced accuracy:", round(balanced_acc, 4))
print("Macro F1:", round(macro_f1, 4))
print("Weighted F1:", round(weighted_f1, 4))


family_names = [
    "workflow_error",
    "constraint_error",
    "tool_use_error",
    "grounding_state_error",
    "reasoning_value_error",
]


print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=[0, 1, 2, 3, 4],
        target_names=family_names,
        zero_division=0,
        digits=4,
    )
)


print("\nConfusion matrix:")
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=[0, 1, 2, 3, 4],
)

cm_df = pd.DataFrame(
    cm,
    index=family_names,
    columns=family_names,
)

display(cm_df)

Accuracy: 0.331
Balanced accuracy: 0.4452
Macro F1: 0.3146
Weighted F1: 0.2596

Classification report:
                       precision    recall  f1-score   support

       workflow_error     0.3333    0.0362    0.0654       138
     constraint_error     0.5182    0.8143    0.6333        70
       tool_use_error     0.2041    0.2632    0.2299        38
grounding_state_error     0.1868    0.5667    0.2810        30
reasoning_value_error     0.2727    0.5455    0.3636        11

             accuracy                         0.3310       287
            macro avg     0.3030    0.4452    0.3146       287
         weighted avg     0.3437    0.3310    0.2596       287


Confusion matrix:


,workflow_error,constraint_error,tool_use_error,grounding_state_error,reasoning_value_error
workflow_error,5,34,31,59,9
constraint_error,2,57,6,4,1
tool_use_error,8,8,10,8,4
grounding_state_error,0,9,2,17,2
reasoning_value_error,0,2,0,3,6


In [7]:
y_prob = classifier.predict_proba(
    X_test_text_embeddings
)

confidence = y_prob.max(axis=1)

confidence_summary = pd.Series(
    confidence
).describe(
    percentiles=[
        0.1,
        0.25,
        0.5,
        0.75,
        0.9,
    ]
)

confidence_summary

count    287.000000
mean       0.493767
std        0.153282
min        0.231283
10%        0.306864
25%        0.377557
50%        0.473850
75%        0.594305
90%        0.700497
max        0.945467
dtype: float64

In [9]:
eval_df = transformer_test_df.copy()

eval_df["y_true"] = y_test.to_numpy()
eval_df["y_pred"] = y_pred
eval_df["confidence"] = confidence

eval_df["correct"] = (
    eval_df["y_true"]
    == eval_df["y_pred"]
)

eval_df[
    ~eval_df["correct"]
].sort_values(
    "confidence",
    ascending=False,
)[[
    "dataset",
    "current_role",
    "failure_family",
    "y_true",
    "y_pred",
    "confidence",
]].head(30)

,dataset,current_role,failure_family,y_true,y_pred,confidence
229,C,ASSISTANT,grounding_state_error,3,4,0.914410
200,C,ASSISTANT,workflow_error,0,3,0.813404
108,B,TOOL_CALL,workflow_error,0,1,0.777759
178,B,TOOL_CALL,workflow_error,0,2,0.752913
188,B,TOOL_CALL,workflow_error,0,2,0.752913
168,B,TOOL_CALL,workflow_error,0,2,0.752913
201,C,TOOL_CALL,workflow_error,0,3,0.746056
203,C,ASSISTANT,workflow_error,0,3,0.743217
192,C,TOOL_CALL,workflow_error,0,3,0.726702
227,C,ASSISTANT,workflow_error,0,4,0.722397


In [10]:
classifier_unweighted = LogisticRegression(
    max_iter=5000,
    C=1.0,
    random_state=42,
)

classifier_unweighted.fit(
    X_train_text_embeddings,
    y_train,
)

pred_unweighted = classifier_unweighted.predict(
    X_test_text_embeddings,
)

print(
    classification_report(
        y_test,
        pred_unweighted,
        labels=[0, 1, 2, 3, 4],
        target_names=family_names,
        digits=4,
        zero_division=0,
    )
)

print(
    "Accuracy:",
    accuracy_score(
        y_test,
        pred_unweighted,
    )
)

print(
    "Balanced accuracy:",
    balanced_accuracy_score(
        y_test,
        pred_unweighted,
    )
)

print(
    "Macro F1:",
    f1_score(
        y_test,
        pred_unweighted,
        average="macro",
    )
)

                       precision    recall  f1-score   support

       workflow_error     0.5537    0.4855    0.5174       138
     constraint_error     0.6026    0.6714    0.6351        70
       tool_use_error     0.2692    0.1842    0.2188        38
grounding_state_error     0.2364    0.4333    0.3059        30
reasoning_value_error     0.8571    0.5455    0.6667        11

             accuracy                         0.4878       287
            macro avg     0.5038    0.4640    0.4688       287
         weighted avg     0.5064    0.4878    0.4902       287

Accuracy: 0.4878048780487805
Balanced accuracy: 0.4639868445818102
Macro F1: 0.4687617344234991


In [11]:
weights = [
    None,

    "balanced",

    {
        0: 1.0,
        1: 1.2,
        2: 1.3,
        3: 1.3,
        4: 2.0,
    },

    {
        0: 1.0,
        1: 1.1,
        2: 1.2,
        3: 1.2,
        4: 1.5,
    },
]

In [12]:
results = []

for weight in weights:

    clf = LogisticRegression(
        class_weight=weight,
        max_iter=5000,
        C=1.0,
        random_state=42,
    )

    clf.fit(
        X_train_text_embeddings,
        y_train,
    )

    pred = clf.predict(
        X_test_text_embeddings,
    )

    results.append({
        "class_weight": str(weight),
        "accuracy": accuracy_score(
            y_test,
            pred,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_test,
            pred,
        ),
        "macro_f1": f1_score(
            y_test,
            pred,
            average="macro",
        ),
        "weighted_f1": f1_score(
            y_test,
            pred,
            average="weighted",
        ),
        "workflow_recall": (
            (pred[y_test.to_numpy() == 0] == 0)
            .mean()
        ),
    })

pd.DataFrame(results)

,class_weight,accuracy,balanced_accuracy,macro_f1,weighted_f1,workflow_recall
0,None,0.487805,0.463987,0.468762,0.490172,0.485507
1,balanced,0.331010,0.445159,0.314641,0.259646,0.036232
2,"{0: 1.0, 1: 1.2, 2: 1.3, 3: 1.3, 4: 2.0}",0.439024,0.462169,0.447197,0.440407,0.340580
3,"{0: 1.0, 1: 1.1, 2: 1.2, 3: 1.2, 4: 1.5}",0.470383,0.469995,0.464992,0.475013,0.413043


In [ ]:
error_analysis = transformer_test_df.copy()

error_analysis["true"] = y_test.to_numpy()
error_analysis["pred"] = y_pred
error_analysis["confidence"] = confidence

workflow_errors = (
    error_analysis[
        (error_analysis["true"] == 0)
        & (error_analysis["pred"] != 0)
    ]
    .sort_values(
        "confidence",
        ascending=False,
    )
)

workflow_errors.head(50)

,dataset,group_id,trajectory_index,message_index,current_role,context_text,current_text,previous_messages,previous_tool_calls,previous_tool_results,previous_user_messages,previous_assistant_messages,failure_family,family_label,transformer_text,true,pred,confidence
200,C,c_15,15,70,ASSISTANT,[TOOL_CALL]\n\nls({})\n\n[TOOL_RESULT name=ls]...,[ASSISTANT]\nPerfect! I have successfully sort...,65,32,32,0,1,workflow_error,0,[CONTEXT]\n[TOOL_CALL]\n\nls({})\n\n[TOOL_RESU...,0,3,0.813404
108,B,b_48,48,29,TOOL_CALL,[ASSISTANT]\nThanks for the clarification. I c...,"[TOOL_CALL]\n\ntransfer_to_human_agents({""summ...",21,8,8,0,7,workflow_error,0,[CONTEXT]\n[ASSISTANT]\nThanks for the clarifi...,0,1,0.777759
168,B,b_230,230,25,TOOL_CALL,[TOOL_CALL]\n\ncheck_vpn_status({})\n\n[TOOL_R...,[TOOL_CALL]\n\nrun_speed_test({}),23,11,11,0,1,workflow_error,0,[CONTEXT]\n[TOOL_CALL]\n\ncheck_vpn_status({})...,0,2,0.752913
178,B,b_235,235,25,TOOL_CALL,[TOOL_CALL]\n\ncheck_vpn_status({})\n\n[TOOL_R...,[TOOL_CALL]\n\nrun_speed_test({}),23,11,11,0,1,workflow_error,0,[CONTEXT]\n[TOOL_CALL]\n\ncheck_vpn_status({})...,0,2,0.752913
188,B,b_240,240,25,TOOL_CALL,[TOOL_CALL]\n\ncheck_vpn_status({})\n\n[TOOL_R...,[TOOL_CALL]\n\nrun_speed_test({}),23,11,11,0,1,workflow_error,0,[CONTEXT]\n[TOOL_CALL]\n\ncheck_vpn_status({})...,0,2,0.752913
201,C,c_15,15,73,TOOL_CALL,"[TOOL_RESULT name=ls]\n{""current_directory_con...","[TOOL_CALL]\n\nwc({""file_name"": ""2024_research...",66,32,32,0,2,workflow_error,0,"[CONTEXT]\n[TOOL_RESULT name=ls]\n{""current_di...",0,3,0.746056
203,C,c_15,15,77,ASSISTANT,"[TOOL_CALL]\n\ncat({""file_name"": ""2024_researc...","[ASSISTANT]\nYes, I can see the three lines cl...",70,34,34,0,2,workflow_error,0,"[CONTEXT]\n[TOOL_CALL]\n\ncat({""file_name"": ""2...",0,3,0.743217
192,C,c_15,15,54,TOOL_CALL,"[TOOL_CALL]\n\necho({""content"": ""Line 3: Exper...","[TOOL_CALL]\n\ntouch({""file_name"": ""temp_sorte...",49,24,24,0,1,workflow_error,0,"[CONTEXT]\n[TOOL_CALL]\n\necho({""content"": ""Li...",0,3,0.726702
227,C,c_80,80,14,ASSISTANT,[TOOL_CALL]\n\nestimate_drive_feasibility_by_m...,"[ASSISTANT]\nBased on the assessment, your veh...",9,4,4,0,1,workflow_error,0,[CONTEXT]\n[TOOL_CALL]\n\nestimate_drive_feasi...,0,4,0.722397
32,A,a_54,54,14,ASSISTANT,"[TOOL_CALL]\n\nsearch({""query"": ""\""6 March 193...",[ASSISTANT]\n<think>There is insufficient cont...,9,3,3,0,3,workflow_error,0,"[CONTEXT]\n[TOOL_CALL]\n\nsearch({""query"": ""\""...",0,3,0.719419


In [18]:
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
)

experiments = {
    "unweighted": None,
    "mild_weight": {
        0: 1.0,
        1: 1.1,
        2: 1.2,
        3: 1.2,
        4: 1.5,
    },
}

for name, weights in experiments.items():

    clf = LinearSVC(
        C=1.0,
        class_weight=weights,
        max_iter=20000,
        random_state=42,
    )

    clf.fit(
        X_train_text_embeddings,
        y_train,
    )

    pred = clf.predict(
        X_test_text_embeddings,
    )

    print("\n", name)
    print("Accuracy:",
          accuracy_score(y_test, pred))
    print("Balanced accuracy:",
          balanced_accuracy_score(y_test, pred))
    print("Macro F1:",
          f1_score(y_test, pred, average="macro"))

    print(
        classification_report(
            y_test,
            pred,
            target_names=family_names,
            digits=4,
            zero_division=0,
        )
    )


 unweighted
Accuracy: 0.47735191637630664
Balanced accuracy: 0.4572373621801539
Macro F1: 0.4602757643462338
                       precision    recall  f1-score   support

       workflow_error     0.5575    0.4565    0.5020       138
     constraint_error     0.5976    0.7000    0.6447        70
       tool_use_error     0.2500    0.1842    0.2121        38
grounding_state_error     0.2105    0.4000    0.2759        30
reasoning_value_error     0.8571    0.5455    0.6667        11

             accuracy                         0.4774       287
            macro avg     0.4946    0.4572    0.4603       287
         weighted avg     0.5018    0.4774    0.4811       287


 mild_weight
Accuracy: 0.445993031358885
Balanced accuracy: 0.45182164898412036
Macro F1: 0.44991849949755247
                       precision    recall  f1-score   support

       workflow_error     0.5306    0.3768    0.4407       138
     constraint_error     0.5976    0.7000    0.6447        70
       tool_use_err

In [25]:
train_group_set = set(groups_train)
test_group_set = set(groups_test)

print("Group overlap:", len(train_group_set & test_group_set))

Group overlap: 0


In [26]:
train_mask = taxonomy_df["group_id"].isin(
    train_group_set
)

test_mask = taxonomy_df["group_id"].isin(
    test_group_set
)

In [27]:
print(
    "Train rows:",
    train_mask.sum()
)

print(
    "Test rows:",
    test_mask.sum()
)

print(
    "Overlap rows:",
    (train_mask & test_mask).sum()
)

Train rows: 1489
Test rows: 287
Overlap rows: 0


In [28]:
assert train_mask.sum() == len(y_train)
assert test_mask.sum() == len(y_test)

assert not (train_mask & test_mask).any()

assert (
    train_mask.sum()
    + test_mask.sum()
    == len(taxonomy_df)
)

print("✓ Exact taxonomy split recovered.")

✓ Exact taxonomy split recovered.


In [29]:
texts = {
    "current_only": (
        taxonomy_df["current_text"]
        .fillna("")
        .astype(str)
        .tolist()
    ),

    "context_only": (
        taxonomy_df["context_text"]
        .fillna("")
        .astype(str)
        .tolist()
    ),

    "context_current": (
        "[CONTEXT]\n"
        + taxonomy_df["context_text"]
        .fillna("")
        .astype(str)
        + "\n\n[CURRENT]\n"
        + taxonomy_df["current_text"]
        .fillna("")
        .astype(str)
    ).tolist(),
}

In [32]:
embeddings = {}

for name, text_list in texts.items():

    print(f"Embedding {name}...")

    embeddings[name] = reason_model.encode(
        text_list,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

Embedding current_only...


Batches:   0%|          | 0/28 [00:00<?, ?it/s]

Embedding context_only...


Batches:   0%|          | 0/28 [00:00<?, ?it/s]

Embedding context_current...


Batches:   0%|          | 0/28 [00:00<?, ?it/s]

In [33]:
embedding_splits = {}

for name, X in embeddings.items():

    embedding_splits[name] = {
        "train": X[train_mask.to_numpy()],
        "test": X[test_mask.to_numpy()],
    }

    print(
        name,
        embedding_splits[name]["train"].shape,
        embedding_splits[name]["test"].shape,
    )

current_only (1489, 384) (287, 384)
context_only (1489, 384) (287, 384)
context_current (1489, 384) (287, 384)


In [35]:
import numpy as np

y_train_from_df = (
    taxonomy_df.loc[
        train_mask,
        "family_label",
    ]
    .to_numpy()
)

y_test_from_df = (
    taxonomy_df.loc[
        test_mask,
        "family_label",
    ]
    .to_numpy()
)

print(
    np.array_equal(
        y_train_from_df,
        y_train.to_numpy(),
    )
)

print(
    np.array_equal(
        y_test_from_df,
        y_test.to_numpy(),
    )
)

True
True


In [36]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
)
import pandas as pd


family_names = [
    "workflow_error",
    "constraint_error",
    "tool_use_error",
    "grounding_state_error",
    "reasoning_value_error",
]


ablation_results = []

for name, split in embedding_splits.items():

    clf = LogisticRegression(
        max_iter=5000,
        random_state=42,
    )

    clf.fit(
        split["train"],
        y_train,
    )

    pred = clf.predict(
        split["test"]
    )

    accuracy = accuracy_score(
        y_test,
        pred,
    )

    balanced_acc = balanced_accuracy_score(
        y_test,
        pred,
    )

    macro_f1 = f1_score(
        y_test,
        pred,
        average="macro",
    )

    weighted_f1 = f1_score(
        y_test,
        pred,
        average="weighted",
    )

    ablation_results.append({
        "input": name,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    })

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    print(
        classification_report(
            y_test,
            pred,
            labels=[0, 1, 2, 3, 4],
            target_names=family_names,
            digits=4,
            zero_division=0,
        )
    )


ablation_results_df = (
    pd.DataFrame(ablation_results)
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(ablation_results_df)


current_only
                       precision    recall  f1-score   support

       workflow_error     0.6230    0.5507    0.5846       138
     constraint_error     0.5584    0.6143    0.5850        70
       tool_use_error     0.3030    0.2632    0.2817        38
grounding_state_error     0.2708    0.4333    0.3333        30
reasoning_value_error     0.8571    0.5455    0.6667        11

             accuracy                         0.5157       287
            macro avg     0.5225    0.4814    0.4903       287
         weighted avg     0.5370    0.5157    0.5215       287


context_only
                       precision    recall  f1-score   support

       workflow_error     0.5827    0.5362    0.5585       138
     constraint_error     0.5208    0.7143    0.6024        70
       tool_use_error     0.4000    0.1053    0.1667        38
grounding_state_error     0.2449    0.4000    0.3038        30
reasoning_value_error     0.8000    0.3636    0.5000        11

             accuracy 

,input,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,current_only,0.515679,0.481391,0.490268,0.521487
1,context_current,0.494774,0.474923,0.477660,0.498444
2,context_only,0.501742,0.423883,0.426273,0.488459


Yes. At this point you have enough evidence to close the **taxonomy construction + classical baseline/exploration stage** and state several concrete findings.

## Final summary of this research stage

The work so far answers three questions:

> **Can heterogeneous agent failures be mapped into a reusable taxonomy? Is that taxonomy learnable? And how much does trajectory context help simple models predict it?**

The answer is: **yes, the taxonomy is learnable, but the current failure step itself carries more immediately usable signal for frozen embeddings than adding raw trajectory context.**

### 1. You converted heterogeneous annotations into a common taxonomy

You started with three datasets whose annotations described failures in very different, often domain-specific language. Directly using those descriptions as labels would have produced a huge, sparse taxonomy.

Through semantic normalization, clustering, matching, prototype/nearest-neighbor assignment, and manual inspection of uncertain cases, you reduced these into five reusable families:

| Failure family          | Meaning                                                                                                 |
| ----------------------- | ------------------------------------------------------------------------------------------------------- |
| `workflow_error`        | Wrong sequence, repeated/irrelevant action, missing action, unresolved previous error                   |
| `constraint_error`      | Violates a task, user, system, policy, or other constraint                                              |
| `tool_use_error`        | Wrong/unavailable tool, malformed call, missing/wrong arguments, authentication/tool execution problems |
| `grounding_state_error` | Unsupported claims, hallucinated values, incorrect state/location, misinterpreted tool results          |
| `reasoning_value_error` | Incorrect reasoning, factual/value/unit-conversion errors                                               |

You retained **1,776 labeled failure examples across 419 trajectories**.

The distribution is imbalanced:

```text
workflow_error          798
constraint_error        387
tool_use_error          275
grounding_state_error   274
reasoning_value_error    42
```

That imbalance is real and itself informative: workflow failures dominate these agent trajectories.

---

## 2. The taxonomy crosses dataset boundaries, but dataset distributions differ

All major families occur across datasets A/B/C, although their proportions differ substantially.

For example, `workflow_error` is:

```text
A: 42.68%
B: 50.23%
C: 35.06%
```

while grounding/state failures are:

```text
A: 27.44%
B:  8.67%
C: 25.23%
```

So the taxonomy is **shared**, but the domains have different failure distributions.

That's an important result.

It means you haven't simply discovered three dataset identifiers disguised as labels. But it also means domain shift is going to matter.

---

## 3. You built a leakage-safe evaluation protocol

This is an important methodological part of the project.

Individual rows from the same trajectory cannot be randomly divided between train and test because adjacent failures share substantial context.

You therefore split on `group_id`:

```text
335 training trajectories
84 test trajectories

overlap = 0
```

giving:

```text
Train: 1489 examples
Test:   287 examples
```

You also verified that the reconstructed embedding split exactly matches the labels:

```text
train alignment: True
test alignment:  True
```

So your evaluation is measuring generalization to **unseen trajectories**, rather than memorization of neighboring steps.

---

## 4. A trivial baseline cannot solve the taxonomy

The majority class is `workflow_error`.

Its baseline was approximately:

```text
Accuracy:  ~0.44
Macro F1:  ~0.123
```

This is why accuracy alone is misleading.

A model can obtain ~44% accuracy by predicting `workflow_error` everywhere while essentially learning nothing about the taxonomy.

Therefore **Macro F1 should be one of your primary metrics**.

---

## 5. The taxonomy contains learnable semantic signal

Your classical models substantially beat the majority baseline on Macro F1.

The strongest frozen-embedding ablation is now:

| Input             |  Accuracy | Balanced Acc. |  Macro F1 |
| ----------------- | --------: | ------------: | --------: |
| **Current only**  | **0.516** |     **0.481** | **0.490** |
| Context + current |     0.495 |         0.475 |     0.478 |
| Context only      |     0.502 |         0.424 |     0.426 |
| Majority baseline |     ~0.44 |         ~0.20 |    ~0.123 |

This is probably the strongest result of this stage.

Your automatically derived taxonomy is **not arbitrary**.

A simple frozen 384-dimensional semantic representation followed by a linear classifier can predict the five failure families with:

> **Macro F1 ≈ 0.49 on unseen trajectories.**

That is roughly **4× the majority baseline Macro F1**.

So there is meaningful semantic structure in the taxonomy.

---

# 6. The ablation gave an unexpected and useful result

Your original hypothesis could reasonably have been:

```text
context + current > current > context
```

But you actually found:

```text
current_only       Macro F1 = .490
context_current    Macro F1 = .478
context_only       Macro F1 = .426
```

So **raw additional context does not improve frozen embeddings**.

In fact, it slightly hurts.

That's not a failed experiment. It's an interesting finding.

### Current action contains strong local failure signals

`current_only` performs best:

```text
Accuracy = .516
Macro F1 = .490
```

That suggests many failure families manifest directly in the erroneous action itself.

For example:

```text
book_flight(...)
```

with missing parameters may directly signal a tool-use error.

Or:

```text
"I've successfully booked your flight"
```

may contain language characteristic of state/grounding or workflow errors.

The model doesn't always need the full trajectory to extract useful information.

---

# 7. Context itself still contains predictive information

Don't interpret the result as:

> "Context doesn't matter."

That's not what the experiment shows.

`context_only` gets:

```text
Accuracy = .502
Macro F1 = .426
```

without even seeing the failure-producing current step.

That's actually remarkable.

It means the trajectory leading into an error already contains information about **what kind of error is likely to happen next**.

There may be recurring failure patterns such as:

```text
bad state
    ↓
incorrect tool sequence
    ↓
repeated recovery attempt
    ↓
workflow failure
```

or:

```text
incorrect assumption
    ↓
tool result
    ↓
unsupported conclusion
```

So failures aren't independent isolated events. They occur in characteristic trajectory structures.

---

# 8. But naïvely concatenating context doesn't help

This is the more precise conclusion.

You gave a frozen embedding model:

```text
[CONTEXT]
large trajectory...

[CURRENT]
specific erroneous action
```

and performance dropped from:

```text
.490 → .478 Macro F1
```

That's small, but clearly not an improvement.

A likely explanation is **representation dilution**.

A sentence embedding compresses the entire input into one 384-dimensional vector:

```text
long context + current action
             ↓
        384 numbers
```

The important relationship may be something like:

> tool result says booking failed
> BUT
> assistant says booking succeeded

A generic embedding isn't explicitly trained to perform that comparison. Adding more text can therefore dilute the highly discriminative local signal in `current_text`.

This gives you a strong motivation for the transformer experiment.

---

# 9. Different failure families have very different difficulty

Your best `current_only` model gives:

| Family                  |       F1 |
| ----------------------- | -------: |
| `reasoning_value_error` | **.667** |
| `constraint_error`      | **.585** |
| `workflow_error`        | **.585** |
| `grounding_state_error` |     .333 |
| `tool_use_error`        | **.282** |

This pattern has been fairly stable across your LR/SVC experiments.

That's another major finding.

### Easier

`workflow_error` and `constraint_error` have relatively strong identifiable signals.

`reasoning_value_error` looks excellent, although its test support is only **11**, so you should be very cautious about interpreting `.667` as stable.

### Harder

`tool_use_error`:

```text
precision .303
recall    .263
F1        .282
```

and `grounding_state_error`:

```text
precision .271
recall    .433
F1        .333
```

remain difficult.

Those categories often require knowing:

* what tools are available;
* what arguments are required;
* what the previous tool returned;
* what state should currently hold;
* whether the assistant's claim is supported by that state.

Those are fundamentally more relational than simply recognizing the semantics of the current sentence.

---

# 10. Class balancing was not beneficial

You also learned something important experimentally.

Automatic:

```python
class_weight="balanced"
```

destroyed workflow recall and produced:

```text
Accuracy = .331
Macro F1 = .315
```

Milder manual weighting improved that, but still didn't beat unweighted training.

So the evidence currently says:

> **Do not automatically compensate for this taxonomy's class imbalance with inverse-frequency class weights.**

At least for these linear models, the correction is too aggressive.

That's worth documenting because it's a real negative result.

---

# 11. Cross-dataset generalization is substantially harder

Your earlier leave-one-dataset-out experiments were much worse:

```text
A+B → C    Macro F1 ≈ .177
A+C → B    Macro F1 ≈ .187
B+C → A    Macro F1 ≈ .239
```

compared with approximately:

```text
grouped mixed-domain split
Macro F1 ≈ .49
```

This is a **very important research result**.

It says there are two different notions of generalization:

### Unseen-trajectory generalization

> I've never seen this trajectory, but I've trained on examples from its broader environments.

Your model can do this moderately well.

### Unseen-domain generalization

> I've never trained on this dataset/environment at all.

Performance collapses.

Therefore the taxonomy appears cross-domain, but the **linguistic and behavioral realization of each failure family remains domain dependent**.

That's potentially one of the main findings of the eventual project.

---

# 12. What you've demonstrated so far

You can now make a defensible claim:

> **Heterogeneous agent failure annotations can be normalized into a compact cross-domain taxonomy whose high-level failure families are learnable from agent behavior. Simple semantic representations generalize substantially above trivial baselines to unseen trajectories, but generalization to unseen domains remains difficult.**

And the ablation adds:

> **The erroneous current action is more predictive than raw trajectory context for frozen embedding classifiers. Context alone nevertheless carries substantial predictive information, suggesting that failure modes have temporal structure. Naïve context concatenation does not improve performance, motivating models capable of explicitly learning relationships between trajectory state and current action.**

That's much stronger than merely saying "my classifier got 51% accuracy."

---

# 13. What this stage does NOT establish

You should also be careful about what hasn't been demonstrated yet.

You haven't shown that:

* `.49` Macro F1 is close to the maximum possible;
* transformers will outperform embeddings;
* context is fundamentally unnecessary;
* the five-family taxonomy is optimal;
* the fine-grained taxonomy is reliably predictable;
* the model generalizes strongly to completely new agent domains.

Those become subsequent experiments.

---

# Where the project stands

Conceptually, you've completed:

```text
Raw heterogeneous datasets
          ↓
Failure annotation extraction
          ↓
Semantic normalization
          ↓
Fine-grained failure taxonomy
          ↓
5-family taxonomy
          ↓
Trajectory-safe dataset
          ↓
Majority baseline
          ↓
TF-IDF baseline
          ↓
Frozen embedding baseline
          ↓
Classifier comparison
          ↓
Class-weight ablation
          ↓
Current/context ablation
          ↓
        ✓ HERE
```

The next stage is:

```text
        ✓ HERE
          ↓
Fine-tuned transformer
          ↓
Current vs context+current transformer ablation
          ↓
Cross-dataset transformer evaluation
          ↓
Detailed confusion/error analysis
          ↓
Final conclusions
```

The most important thing now is **not to modify the taxonomy because the classifier makes mistakes**. Freeze the taxonomy and evaluation split. Otherwise the target keeps moving while you're evaluating models.

You now have a legitimate baseline of approximately **0.49 Macro F1 / 0.516 accuracy on unseen trajectories**. Treat that as the number the next representation/model needs to beat.

The transformer experiment should answer the next real research question: **can task-specific representation learning exploit trajectory context in a way that frozen 384-dimensional embeddings cannot?**


Yes—there are actually **two experiments** embedded in what you just did.

### 1. The overall experiment: failure taxonomy classification

A good formal name is:

**Agent Failure Taxonomy Classification**

More specifically:

> **Multi-class classification of agent failure families from trajectory data**

Your target is:

[
f(x) \rightarrow
{\text{workflow},\text{constraint},\text{tool-use},\text{grounding/state},\text{reasoning/value}}
]

where (x) is some representation of an agent step and its history.

### 2. The current/context comparison: input-context ablation

The experiment where you compare:

```text
current_only
context_only
context_current
```

is called an **input ablation study** or, more specifically, a **context ablation study**.

You're asking:

> How much predictive information comes from the current action, the preceding trajectory context, and their combination?

Your result:

| Input             |  Macro F1 |
| ----------------- | --------: |
| current only      | **0.490** |
| context + current |     0.478 |
| context only      |     0.426 |

So a good experiment title would be:

**Context Ablation for Agent Failure Classification**

---

## And yes, your intuition about Logistic Regression is mostly correct

There is an important distinction, though.

The limitation isn't just Logistic Regression. Your pipeline is approximately:

[
\text{context + current}
\rightarrow
\text{frozen embedding}
\rightarrow
\text{Logistic Regression}
]

The embedding model first collapses the entire sequence into one vector:

[
E(\text{context,current}) \in \mathbb{R}^{384}
]

Then Logistic Regression learns:

[
P(y=k|x)=\operatorname{softmax}(W x+b)
]

So LR only sees those **384 numbers**. It doesn't see:

```text
TOOL_RESULT:
booking_status = failed

CURRENT ASSISTANT:
"Your booking was successful!"
```

as two separate pieces of information.

It sees something like:

```text
[-0.14, 0.22, 0.03, ..., 0.19]
```

and draws linear decision boundaries through that embedding space.

### The pattern you actually care about is relational

For many of your categories, classification requires something like:

[
\text{Failure} =
g(\text{previous state},\text{tool result},\text{current action})
]

For example:

```text
previous tool result
    ↓
booking FAILED

current response
    ↓
"Booking completed successfully"

relationship
    ↓
CONTRADICTION

failure
    ↓
grounding/state error
```

The important information isn't necessarily in either text independently.

It's in the **relationship between them**.

That's exactly where your current embedding + LR setup is weak.

---

## But don't conclude "LR is bad"

Your LR experiment is actually doing its job perfectly as a **linear probe**.

That's another useful term for what you're doing:

> **Frozen-embedding linear probing**

You freeze a representation and ask:

> "How linearly separable are my failure families in this representation?"

Your answer is roughly:

> Quite separable—Macro F1 ~0.49—but adding raw context doesn't make them more separable.

That's valuable.

You shouldn't try to make Logistic Regression extremely sophisticated. It's your **baseline/probe**.

---

# The next model should change the representation

I'd make the progression:

```text
1. Majority baseline                 ✓
2. TF-IDF + LR/SVM                   ✓
3. Frozen embedding + LR             ✓
4. Context ablation                  ✓

                ↓

5. Fine-tuned transformer
6. Transformer context ablation
7. Cross-dataset transformer eval
```

A sequence classifier is the natural next experiment.

Conceptually:

```text
[CLS]

[CONTEXT]
user: ...
assistant: ...
tool_call: ...
tool_result: ...

[CURRENT]
assistant: ...

[SEP]
```

Then:

[
\text{Transformer}
(\text{context,current})
\rightarrow h_{\text{CLS}}
\rightarrow \text{classification head}
\rightarrow y
]

Unlike your frozen embedding pipeline, fine-tuning allows the representation itself to change in response to the taxonomy.

---

## Why that experiment is scientifically interesting

You already have a hypothesis from the ablation:

> **H1:** Context contains useful failure information, but frozen pooled embeddings do not represent context-current interactions sufficiently well.

You have evidence supporting the first half because:

[
F1(\text{context-only})=.426 \gg .123
]

So context clearly contains signal.

But:

[
F1(\text{context+current})=.478
<
F1(\text{current})=.490
]

means simply concatenating context doesn't help the frozen representation.

Now test:

[
F1_{\text{Transformer}}(\text{context+current})
\stackrel{?}{>}
F1_{\text{Transformer}}(\text{current})
]

That is a much stronger experiment.

### Possible outcomes

If you get:

```text
Frozen embeddings
current           .490
context+current   .478

Transformer
current           .55
context+current   .64
```

then you have evidence that:

> Context was useful, but exploiting it required **task-specific contextual representation learning**.

That's a strong result.

If instead you get:

```text
Transformer
current           .61
context+current   .59
```

that's also interesting.

It suggests the failure taxonomy is predominantly **locally identifiable** from the erroneous action, and historical context contributes little additional discriminative information.

Either outcome tells you something.

---

## One more model design I'd eventually test

There is an intermediate architecture between your current setup and a full transformer that is particularly relevant to your question.

Instead of:

[
E(\text{context}+\text{current})
]

encode them **separately**:

[
c = E(\text{context})
]

[
a = E(\text{current})
]

and construct:

[
x=[c;a;|c-a|;c\odot a]
]

where (c\odot a) is element-wise interaction.

Then train Logistic Regression or an MLP.

That explicitly gives the classifier some context/current relationship information.

So you could eventually have a very clean architecture comparison:

```text
A. Current embedding → LR
                baseline

B. concat(context,current) embedding → LR
                naïve context

C. separate embeddings + interaction features → MLP
                explicit interaction

D. context/current → fine-tuned Transformer
                learned interaction
```

That would turn your current exploration into a quite coherent model study.

So I'd describe what you've completed as **a context ablation study using frozen-embedding linear probes**, inside the broader **agent failure taxonomy classification** experiment. Your next major experiment should test **task-specific contextual representation learning**, rather than trying to squeeze substantially more performance out of Logistic Regression.
